# Deepfake Detection Evaluation on FaceForensics++
## Re-running Xception Baseline & RGB+FFT Dual-Stream Models

This notebook downloads FaceForensics++ (c23 compression), preprocesses spatial and FFT frequency domain representations, and evaluates both the **Xception baseline** and the **RGB+FFT Dual-Stream model** to obtain:
- **Accuracy**
- **F1-Score**
- **AUC (Area Under ROC Curve)**
- **EER (Equal Error Rate)**

All evaluation metrics use F1-optimal threshold selection on validation data.

### Step 1: Clone Repository & Install Dependencies

In [ ]:
# Mount Google Drive if persistent storage is desired (optional)
# from google.colab import drive
# drive.mount('/content/drive')

# Install dependencies
!pip install -r requirements.txt
!apt-get update && apt-get install -y ffmpeg

### Step 2: Download FaceForensics++ Dataset (Deepfakes + Original)

In [ ]:
# Download a sample subset (e.g. 50 videos) of Deepfakes and Original YouTube videos non-interactively (-y)
!python scripts/download/download_faceforensics.py data/faceforensics_raw -d Deepfakes -c c23 -t videos -n 50 --server EU -y
!python scripts/download/download_faceforensics.py data/faceforensics_raw -d original -c c23 -t videos -n 50 --server EU -y

### Step 3: Dataset Preprocessing (Frame Extraction, MTCNN Face Detection, FFT Maps)

In [ ]:
# Extract frames, faces, and FFT frequency spectrums into data/
!python scripts/preprocess.py --dataset-type faceforensics --videos-dir data/faceforensics_raw --max_videos 100

### Step 4: Evaluate Xception Baseline Model

In [ ]:
!python scripts/evaluate.py --model xception --config config/config.yaml --split test --optimal_threshold --output_dir results

### Step 5: Evaluate RGB+FFT Dual-Stream Model

In [ ]:
!python scripts/evaluate.py --model rgb_fft_dual_stream --config config/config.yaml --split test --optimal_threshold --output_dir results

### Step 6: Summary Table & Log Packaging

In [ ]:
import os
import glob

print("="*60)
print("RESULTS SUMMARY (FaceForensics++ Deepfakes, c23)")
print("="*60)

result_files = glob.glob("results/*_results.txt")
for rf in result_files:
    print(f"\n--- {os.path.basename(rf)} ---")
    with open(rf, 'r') as f:
        print(f.read())

# Package results and logs
!tar -czf faceforensics_eval_results.tar.gz results/ logs/